In [ ]:
# !nvidia-smi

Thu Nov 26 17:02:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# !git clone https://github.com/ursweetheart/prjML.git /content/project

Cloning into '/content/project'...
remote: Enumerating objects: 140, done.
remote: Counting objects: 100% (140/140), done.
remote: Compressing objects: 100% (72/72), done.
remote: Total 140 (delta 29), reused 136 (delta 25), pack-reused 0 (from 0)
Receiving objects: 100% (140/140), 615.56 KiB | 9.47 MiB/s, done.
Resolving deltas: 100% (29/29), done.


In [ ]:
# !pip install -q lm-eval accelerate

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import torch, transformers
print(torch.__version__, transformers.__version__, torch.cuda.is_available())

2.14.1+cu130 5.22.1 True


test with qwen 2.5 0.5B

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
name = "Qwen/Qwen2.5-0.5B"
dev = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float16).to(dev).eval()

config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Load Data

In [ ]:
import json, random, collections, numpy as np
from google.colab import drive
drive.mount('/content/drive')

DATA = "/content/drive/MyDrive/vnhsge.jsonl"     # change to where you uploaded it
rows = [json.loads(l) for l in open(DATA, encoding="utf-8")]
# rows = [r for r in rows if r["split"] == "test"]
random.Random(0).shuffle(rows)
rows = rows[:300]                                # small first run; raise later
print(len(rows), rows[0]["subject"], rows[0]["answer"])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
300 English A


Define the prompt format and the scoring function

In [ ]:
tok.padding_side = "left"        # last position = real last token
tok.truncation_side = "left"     # if too long, cut the START so "Đáp án:" survives
LETTER_IDS = [tok.encode(" " + c, add_special_tokens=False)[-1] for c in "ABCD"]
print("letter ids:", LETTER_IDS)

# define the prompt
def make_prompt(q, opts):
    return (f"Câu hỏi: {q}\nA. {opts[0]}\nB. {opts[1]}\nC. {opts[2]}\nD. {opts[3]}\nĐáp án:")

#
@torch.no_grad()
def score(prompts, bs=16):
    """returns (N,4) probabilities over the displayed letters A-D"""
    out = []
    for i in range(0, len(prompts), bs):
        enc = tok(prompts[i:i+bs], return_tensors="pt", padding=True,
                  truncation=True, max_length=1024).to(dev)
        last = model(**enc, logits_to_keep=1).logits[:, -1, :].float()   # (B, vocab)
        lp = torch.log_softmax(last, -1)[:, LETTER_IDS]                   # (B, 4) log-prob
        out.append(torch.softmax(lp, -1).cpu())                           # normalise over 4
    return torch.cat(out)

letter ids: [362, 425, 356, 422]


opotion permutation

In [ ]:
def run_shift(rows, k):
    """show options rotated by k: displayed letter j holds original option (j+k)%4"""
    prompts, gold_disp = [], []
    for r in rows:
        opts = [r["option_a"], r["option_b"], r["option_c"], r["option_d"]]
        shown = [opts[(j + k) % 4] for j in range(4)]
        prompts.append(make_prompt(r["question"], shown))
        gold_disp.append(("ABCD".index(r["answer"]) - k) % 4)   # where the right answer now sits
    q = score(prompts)                                          # probs over displayed letters
    p_orig = torch.roll(q, shifts=k, dims=1)                    # map back to original option order
    return q, p_orig, np.array(gold_disp)

gold = np.array(["ABCD".index(r["answer"]) for r in rows])
Q, POrig, GD = {}, {}, {}
for k in range(4):
    Q[k], POrig[k], GD[k] = run_shift(rows, k)
    pred = Q[k].argmax(1).numpy()
    print(f"shift {k}: accuracy {(pred == GD[k]).mean():.3f}")

shift 0: accuracy 0.317
shift 1: accuracy 0.370
shift 2: accuracy 0.320
shift 3: accuracy 0.370


measure the position bias

In [ ]:
# 1) which letter does the model *say*, regardless of content?
said = np.concatenate([Q[k].argmax(1).numpy() for k in range(4)])
print("said letter share :", {c: round(float((said == i).mean()), 3) for i, c in enumerate("ABCD")})

# 2) accuracy when the right answer sits at displayed position A/B/C/D
gd_all  = np.concatenate([GD[k] for k in range(4)])
hit_all = np.concatenate([(Q[k].argmax(1).numpy() == GD[k]) for k in range(4)])
print("accuracy by gold position:", {c: round(float(hit_all[gd_all == i].mean()), 3) for i, c in enumerate("ABCD")})

# 3) average over the 4 orders (cancels position bias)
avg = sum(POrig[k] for k in range(4)) / 4
print("permutation-averaged accuracy:", round(float((avg.argmax(1).numpy() == gold).mean()), 3))

# 4) save the cache: this is what A.2 and Part D will read later
np.save("/content/drive/MyDrive/qwen05b_test300.npy",
        np.stack([POrig[k].numpy() for k in range(4)]))   # (4 shifts, N, 4)

said letter share : {'A': 0.576, 'B': 0.178, 'C': 0.147, 'D': 0.098}
accuracy by gold position: {'A': 0.677, 'B': 0.297, 'C': 0.237, 'D': 0.167}
permutation-averaged accuracy: 0.39


# A.1

**load data**

In [ ]:
import json, os, time, collections, numpy as np
from google.colab import drive
drive.mount('/content/drive')

DATA = "/content/drive/MyDrive/vnhsge.jsonl"
OUT  = "/content/drive/MyDrive/cache_qwen05b"          # one folder per model
os.makedirs(OUT, exist_ok=True)

rows = [json.loads(l) for l in open(DATA, encoding="utf-8")]   # keep file order: row i <-> cache row i
print(len(rows), collections.Counter(r["split"] for r in rows))

# metadata saved once, so later steps (A.2 conformal, D.3) can align cache rows with split/subject/answer
meta = [{"id": r["id"], "split": r["split"], "subject": r["subject"], "answer": r["answer"]} for r in rows]
json.dump(meta, open(f"{OUT}/meta.json", "w"), ensure_ascii=False)
gold = np.array(["ABCD".index(r["answer"]) for r in rows])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
15827 Counter({'test': 7918, 'calibration': 7909})


**define scorer function and prompt**

In [ ]:
from tqdm.auto import tqdm
tok.padding_side = "left"
tok.truncation_side = "left"            # if too long, cut the start so "Đáp án:" survives
LETTER_IDS = [tok.encode(" " + c, add_special_tokens=False)[-1] for c in "ABCD"]

def make_prompt(q, opts):
    return f"Câu hỏi: {q}\nA. {opts[0]}\nB. {opts[1]}\nC. {opts[2]}\nD. {opts[3]}\nĐáp án:"

@torch.no_grad()
def score(prompts, bs=16, max_length=1024):
    """(N,4) probs over displayed letters. Sorts by length so a batch has similar-length prompts."""
    order = sorted(range(len(prompts)), key=lambda i: len(prompts[i]))
    res = torch.zeros(len(prompts), 4)
    for s in tqdm(range(0, len(order), bs), leave=False):
        idx = order[s:s+bs]
        enc = tok([prompts[i] for i in idx], return_tensors="pt", padding=True,
                  truncation=True, max_length=max_length).to(dev)
        last = model(**enc, logits_to_keep=1).logits[:, -1, :].float()
        lp = torch.log_softmax(last, -1)[:, LETTER_IDS]
        res[idx] = torch.softmax(lp, -1).cpu()          # put results back in original order
    return res

In [ ]:
lens = [len(tok(make_prompt(r["question"], [r["option_a"], r["option_b"], r["option_c"], r["option_d"]]))["input_ids"]) for r in rows]
lens = np.array(lens)
print("tokens: median", int(np.median(lens)), "| p99", int(np.percentile(lens, 99)), "| max", lens.max())
print("prompts longer than 1024 tokens:", int((lens > 1024).sum()))

tokens: median 95 | p99 638 | max 1121
prompts longer than 1024 tokens: 7


**define the shift function and runs the model through the shift**

In [ ]:
def run_shift(k):
    prompts = []
    for r in rows:
        opts = [r["option_a"], r["option_b"], r["option_c"], r["option_d"]]
        prompts.append(make_prompt(r["question"], [opts[(j + k) % 4] for j in range(4)]))
    return score(prompts).numpy()                       # probs over DISPLAYED letters

for k in range(4):
    f = f"{OUT}/shift{k}.npy"
    if os.path.exists(f):
        print(f"shift {k}: already done, skip"); continue
    t = time.time()
    q = run_shift(k)
    np.save(f, q)
    acc = (q.argmax(1) == (gold - k) % 4).mean()
    print(f"shift {k}: acc {acc:.3f}  ({(time.time()-t)/60:.1f} min)")

  0%|          | 0/990 [00:00<?, ?it/s]

shift 0: acc 0.360  (2.7 min)


  0%|          | 0/990 [00:00<?, ?it/s]

shift 1: acc 0.359  (2.8 min)


  0%|          | 0/990 [00:00<?, ?it/s]

shift 2: acc 0.356  (2.8 min)


  0%|          | 0/990 [00:00<?, ?it/s]

shift 3: acc 0.354  (2.8 min)


print out the results

In [ ]:
Q  = {k: np.load(f"{OUT}/shift{k}.npy") for k in range(4)}
GD = {k: (gold - k) % 4 for k in range(4)}              # displayed position of the right answer
split = np.array([m["split"] for m in meta]); subj = np.array([m["subject"] for m in meta])

# map every shift back to the original option order
POrig = {k: np.roll(Q[k], k, axis=1) for k in range(4)}
avg = sum(POrig.values()) / 4

print("per-shift accuracy :", [round(float((Q[k].argmax(1) == GD[k]).mean()), 3) for k in range(4)])
print("permutation-average:", round(float((avg.argmax(1) == gold).mean()), 3))

said = np.concatenate([Q[k].argmax(1) for k in range(4)])
print("said-letter share  :", {c: round(float((said == i).mean()), 3) for i, c in enumerate("ABCD")})

gd_all  = np.concatenate([GD[k] for k in range(4)])
hit_all = np.concatenate([(Q[k].argmax(1) == GD[k]) for k in range(4)])
print("acc by gold position:", {c: round(float(hit_all[gd_all == i].mean()), 3) for i, c in enumerate("ABCD")})

for s in ["calibration", "test"]:
    m = split == s
    print(f"{s:11s} n={m.sum()}  acc(shift0)={(Q[0].argmax(1)[m] == GD[0][m]).mean():.3f}")
for s in sorted(set(subj)):
    m = subj == s
    print(f"  {s:15s} n={m.sum():4d} acc(avg)={(avg.argmax(1)[m] == gold[m]).mean():.3f}")

per-shift accuracy : [0.36, 0.359, 0.356, 0.354]
permutation-average: 0.406
said-letter share  : {'A': 0.564, 'B': 0.192, 'C': 0.141, 'D': 0.103}
acc by gold position: {'A': 0.676, 'B': 0.311, 'C': 0.249, 'D': 0.194}
calibration n=7909  acc(shift0)=0.362
test        n=7918  acc(shift0)=0.357
  Biology         n=1845 acc(avg)=0.305
  Chemistry       n=1945 acc(avg)=0.318
  CivicEducation  n=1879 acc(avg)=0.553
  English         n=2491 acc(avg)=0.579
  Geography       n=1824 acc(avg)=0.358
  History         n=1971 acc(avg)=0.408
  Mathematics     n=2030 acc(avg)=0.332
  Physics         n=1842 acc(avg)=0.341


**calculate log prob**

In [ ]:
np.set_printoptions(precision=3, suppress=True)
print("câu đầu tiên của shift 0")
print("câu hỏi   :", rows[0]["question"][:80])
print("đáp án đúng:", rows[0]["answer"])
print("xác suất trên 4 phương án (A,B,C,D):", Q[0][0])
print("log-prob  trên 4 phương án (A,B,C,D):", np.log(Q[0][0]))
print("tổng xác suất:", Q[0][0].sum())

câu đầu tiên của shift 0
câu hỏi   : Ở loài gà có kiểu nhiễm sắc thể giới tính là
đáp án đúng: B
xác suất trên 4 phương án (A,B,C,D): [0.395 0.203 0.223 0.178]
log-prob  trên 4 phương án (A,B,C,D): [-0.929 -1.593 -1.499 -1.725]
tổng xác suất: 1.0


# A.2

**scores and the conformal procedure**

In [ ]:
import math, numpy as np, pandas as pd

def lac_scores(P, U=None):
    """s(x,y) = 1 - p(y|x) for every option. Shape (N,4)."""
    return 1.0 - P

def aps_scores(P, U):
    """Randomized APS (Romano et al. 2020): probability mass ranked above option y, plus U * p(y)."""
    order = np.argsort(-P, axis=1)                    # most likely option first
    Ps = np.take_along_axis(P, order, axis=1)
    above = np.cumsum(Ps, axis=1) - Ps                # mass strictly above each option
    S = np.empty_like(P)
    np.put_along_axis(S, order, above + U[:, None] * Ps, axis=1)   # back to A,B,C,D order
    return S

def threshold(cal_scores, alpha):
    n = len(cal_scores)
    k = math.ceil((n + 1) * (1 - alpha))              # the ceil((n+1)(1-alpha))-th smallest score
    return np.inf if k > n else np.sort(cal_scores)[k - 1]

def conformal(P, gold, cal, test, alpha, score_fn, rng):
    """P: (N,4) probs in ORIGINAL option order. cal/test: boolean masks. Returns q_hat, sets (Ntest,4), hit (Ntest,)"""
    nc, nt = cal.sum(), test.sum()
    s_cal = score_fn(P[cal], rng.random(nc))[np.arange(nc), gold[cal]]   # score of the TRUE answer
    q = threshold(s_cal, alpha)
    sets = score_fn(P[test], rng.random(nt)) <= q                        # keep options with score <= q_hat
    hit = sets[np.arange(nt), gold[test]]                                # is the right answer in the set?
    return q, sets, hit

**self-check on fake data**

In [ ]:
# If the probabilities are perfectly calibrated, coverage must be about 1 - alpha.
rng = np.random.default_rng(1)
Pf = rng.dirichlet(np.ones(4), 20000)
gf = np.array([rng.choice(4, p=p) for p in Pf])
c = np.arange(20000) < 10000
for fn in (lac_scores, aps_scores):
    _, _, h = conformal(Pf, gf, c, ~c, 0.1, fn, rng)
    print(fn.__name__, "coverage:", round(h.mean(), 3))
    assert 0.88 < h.mean() < 0.92

lac_scores coverage: 0.901
aps_scores coverage: 0.898


**main results on your official split**

In [ ]:
cal, test = split == "calibration", split == "test"
inputs = {"shift0": POrig[0], "perm-avg": avg}
out = []
for pname, P in inputs.items():
    for alpha in (0.1, 0.05):
        for mname, fn in (("LAC", lac_scores), ("APS", aps_scores)):
            q, sets, hit = conformal(P, gold, cal, test, alpha, fn, np.random.default_rng(0))
            out.append(dict(probs=pname, alpha=alpha, method=mname, q_hat=q, target=1 - alpha,
                            coverage=hit.mean(), mean_size=sets.sum(1).mean(),
                            empty=(sets.sum(1) == 0).mean(), top1_acc=(P[test].argmax(1) == gold[test]).mean()))
print(pd.DataFrame(out).round(3).to_string(index=False))

   probs  alpha method  q_hat  target  coverage  mean_size  empty  top1_acc
  shift0   0.10    LAC  0.816    0.90     0.898      3.217    0.0     0.357
  shift0   0.10    APS  0.907    0.90     0.898      3.364    0.0     0.357
  shift0   0.05    LAC  0.837    0.95     0.948      3.502    0.0     0.357
  shift0   0.05    APS  0.954    0.95     0.950      3.669    0.0     0.357
perm-avg   0.10    LAC  0.778    0.90     0.899      3.103    0.0     0.405
perm-avg   0.10    APS  0.862    0.90     0.891      3.239    0.0     0.405
perm-avg   0.05    LAC  0.801    0.95     0.949      3.408    0.0     0.405
perm-avg   0.05    APS  0.931    0.95     0.948      3.602    0.0     0.405


**coverage per subject**

In [ ]:
q, sets, hit = conformal(avg, gold, cal, test, 0.1, lac_scores, np.random.default_rng(0))
tab = (pd.DataFrame({"subject": subj[test], "hit": hit, "size": sets.sum(1)})
         .groupby("subject").agg(n=("hit", "size"), coverage=("hit", "mean"), mean_size=("size", "mean")).round(3))
print(tab)

                   n  coverage  mean_size
subject                                  
Biology          923     0.907      3.420
Chemistry        973     0.958      3.707
CivicEducation   940     0.846      2.329
English         1246     0.848      2.166
Geography        913     0.875      3.116
History          986     0.829      2.746
Mathematics     1016     0.993      3.943
Physics          921     0.946      3.647


**five random re-splits (mean and SD, as the assignment requires)**

In [26]:
N = len(gold); res = []
for seed in range(5):
    rng = np.random.default_rng(seed)
    c = np.zeros(N, bool); c[rng.permutation(N)[:N // 2]] = True      # random half = calibration
    for alpha in (0.1, 0.05):
        for mname, fn in (("LAC", lac_scores), ("APS", aps_scores)):
            _, sets, hit = conformal(avg, gold, c, ~c, alpha, fn, rng)
            res.append(dict(seed=seed, alpha=alpha, method=mname, coverage=hit.mean(), size=sets.sum(1).mean()))
print(pd.DataFrame(res).groupby(["alpha", "method"])[["coverage", "size"]].agg(["mean", "std"]).round(4))
for a in (0.1, 0.05):
    print(f"alpha={a}: target {1 - a:.3f}, upper bound 1-alpha+1/(n+1) = {1 - a + 1 / (N // 2 + 1):.5f}")

             coverage            size        
                 mean     std    mean     std
alpha method                                 
0.05  APS      0.9504  0.0051  3.6241  0.0218
      LAC      0.9511  0.0031  3.4214  0.0124
0.10  APS      0.9005  0.0046  3.2676  0.0165
      LAC      0.9015  0.0066  3.1094  0.0121
alpha=0.1: target 0.900, upper bound 1-alpha+1/(n+1) = 0.90013
alpha=0.05: target 0.950, upper bound 1-alpha+1/(n+1) = 0.95013


# A.3: group-wise weight quantization

In [27]:
import os, sys, subprocess
assert os.path.exists("/content/project"), "the repo is not cloned in this runtime: run the clone cell at the top first"
subprocess.run(["git", "-C", "/content/project", "pull", "-q"], check=False)
sys.path.insert(0, "/content/project/src")
from machinelearningproject.quantizer import quantize, dequantize, packed_bytes, _self_test
_self_test()
print("quantizer self-test passed")

quantizer self-test passed


**per-layer reconstruction error of Qwen2.5-0.5B: INT8 and INT4, symmetric and asymmetric, group size 64**

In [28]:
import torch.nn as nn
GROUP = 64
SETTINGS = [(8, True), (8, False), (4, True), (4, False)]
print("tie_word_embeddings:", model.config.tie_word_embeddings, "(lm_head shares weights with the embedding table, so it is skipped)")

def layer_errors(m, bits, sym, skip=("lm_head",)):
    rows = []
    for name, mod in m.named_modules():
        if isinstance(mod, nn.Linear) and not name.endswith(skip):
            w = mod.weight.data
            t = quantize(w, bits, GROUP, sym)
            err = dequantize(t) - w.float()
            rows.append(dict(layer=name, kind=name.split(".")[-1], rmse=err.pow(2).mean().sqrt().item(),
                             rel=(err.norm() / w.float().norm()).item(),
                             mse_over_step2_12=(err.pow(2).mean() / (t.scale ** 2 / 12).mean()).item(),
                             packed_MB=packed_bytes(t) / 1e6, fp16_MB=w.numel() * 2 / 1e6))
    return pd.DataFrame(rows)

results = {s: layer_errors(model, *s) for s in SETTINGS}
tbl = []
for (bits, sym), df in results.items():
    tbl.append(dict(setting=f"INT{bits} {'sym' if sym else 'asym'}", layers=len(df), mean_rel_err=df.rel.mean(),
                    max_rel_err=df.rel.max(), mse_ratio_min=df.mse_over_step2_12.min(),
                    mse_ratio_max=df.mse_over_step2_12.max(), packed_MB=df.packed_MB.sum(), fp16_MB=df.fp16_MB.sum()))
print(pd.DataFrame(tbl).round(4).to_string(index=False))

tie_word_embeddings: True (lm_head shares weights with the embedding table, so it is skipped)
  setting  layers  mean_rel_err  max_rel_err  mse_ratio_min  mse_ratio_max  packed_MB  fp16_MB
 INT8 sym     168        0.0065       0.0098         0.9726         0.9898   369.0086 715.6531
INT8 asym     168        0.0057       0.0080         0.9930         1.0115   374.5997 715.6531
 INT4 sym     168        0.1179       0.1736         0.9281         0.9910   190.0954 715.6531
INT4 asym     168        0.0970       0.1340         0.9672         1.0054   192.8909 715.6531


**error by layer type, and the first transformer block in detail (INT4 symmetric)**

In [29]:
df = results[(4, True)]
print(df.groupby("kind")[["rmse", "rel"]].mean().round(5))
print()
print(df.head(7).round(5).to_string(index=False))          # the 7 linear layers of block 0
for (bits, sym), d in results.items():
    d.to_csv(f"{OUT}/quant_error_int{bits}_{'sym' if sym else 'asym'}.csv", index=False)
print("saved one CSV per setting to", OUT)

              rmse      rel
kind                       
down_proj  0.00218  0.11761
gate_proj  0.00249  0.11500
k_proj     0.00333  0.12174
o_proj     0.00204  0.11079
q_proj     0.00295  0.11859
up_proj    0.00221  0.11439
v_proj     0.00253  0.12685

                          layer      kind    rmse     rel  mse_over_step2_12  packed_MB  fp16_MB
model.layers.0.self_attn.q_proj    q_proj 0.00835 0.12328            0.98769    0.42650  1.60563
model.layers.0.self_attn.k_proj    k_proj 0.00899 0.10997            0.98924    0.06093  0.22938
model.layers.0.self_attn.v_proj    v_proj 0.00139 0.11948            0.98733    0.06093  0.22938
model.layers.0.self_attn.o_proj    o_proj 0.00153 0.11431            0.98599    0.42650  1.60563
   model.layers.0.mlp.gate_proj gate_proj 0.00260 0.11098            0.98437    2.31526  8.71629
     model.layers.0.mlp.up_proj   up_proj 0.00208 0.11190            0.98425    2.31526  8.71629
   model.layers.0.mlp.down_proj down_proj 0.00218 0.11780           

# A.4: speculative decoding

**the algorithm** (`target_probs` and `draft_probs` return one next-token distribution for every position of the input).
It has no KV cache, so it is slow: it is built to check correctness. Speed measurements belong to D.4.

In [30]:
import numpy as np, torch, itertools, collections
from scipy.stats import chi2

def _sample(p, rng):
    c = np.cumsum(p, dtype=np.float64)                 # no need for p to sum to exactly 1
    return min(int(np.searchsorted(c, rng.random() * c[-1])), len(p) - 1)

def speculative_decode(target_probs, draft_probs, prefix, n_new, gamma, rng):
    """Draft gamma tokens with the small model, score them all with ONE target pass, accept each with
    probability min(1, p/q); on the first rejection resample from (p - q)+ and stop; if all are accepted,
    take one extra token from the target."""
    ids = list(prefix)
    stats = dict(target_calls=0, drafted=0, accepted=0)
    while len(ids) < len(prefix) + n_new:
        cur, qs, xs = list(ids), [], []
        for _ in range(gamma):
            q = draft_probs(cur)[-1]
            x = _sample(q, rng)
            qs.append(q); xs.append(x); cur.append(x)
        P = target_probs(cur)                          # row j = distribution of the token after position j
        stats["target_calls"] += 1; stats["drafted"] += gamma
        n = len(ids)
        for i, x in enumerate(xs):
            p, q = P[n - 1 + i], qs[i]
            if rng.random() < min(1.0, p[x] / q[x]):
                ids.append(x); stats["accepted"] += 1
            else:
                r = np.maximum(p - q, 0.0)
                ids.append(_sample(r if r.sum() > 0 else p, rng))
                break
        else:
            ids.append(_sample(P[-1], rng))            # every draft accepted: free extra token
    return ids[:len(prefix) + n_new], stats            # cutting the tail keeps the distribution

**test 1, toy language models with an exactly known distribution: does the output follow the target?**

In [31]:
def make_tables(V, seed):
    r = np.random.default_rng(seed)
    Tp = r.dirichlet(np.ones(V), size=V)                       # target: row i = P(next token | last token i)
    Tq = 0.6 * Tp + 0.4 * r.dirichlet(np.ones(V), size=V)      # draft: related to the target but different
    return Tp, Tq

def exact_dist(T, prefix, n, V):
    out = {}
    for seq in itertools.product(range(V), repeat=n):
        p, last = 1.0, prefix[-1]
        for t in seq:
            p *= T[last, t]; last = t
        out[seq] = p
    return out

def chi_test(counts, probs, N):
    seqs = list(probs)
    obs = np.array([counts.get(s, 0) for s in seqs], float)
    exp = np.array([probs[s] for s in seqs]) * N
    keep = exp >= 5
    stat = (((obs - exp) ** 2 / exp)[keep]).sum()
    pooled = (~keep).any()
    if pooled:
        stat += (obs[~keep].sum() - exp[~keep].sum()) ** 2 / exp[~keep].sum()
    dof = int(keep.sum() + pooled - 1)
    return stat, chi2.sf(stat, dof)

V, n_new, gamma, N = 5, 3, 3, 60000
Tp, Tq = make_tables(V, 0)
tp = lambda ids: Tp[np.array(ids)]
tq = lambda ids: Tq[np.array(ids)]
prefix = [0]
exact = exact_dist(Tp, prefix, n_new, V)

rng = np.random.default_rng(0)
cnt = collections.Counter(tuple(speculative_decode(tp, tq, prefix, n_new, gamma, rng)[0][1:]) for _ in range(N))
stat, pv = chi_test(cnt, exact, N)
print(f"speculative output vs target distribution: chi2 = {stat:.1f}, p-value = {pv:.3f}   (a large p-value = same distribution)")

# control: sampling from the draft alone must be REJECTED, which shows the test can detect a wrong distribution
cnt_d = collections.Counter(tuple(speculative_decode(tq, tq, prefix, n_new, gamma, rng)[0][1:]) for _ in range(N))
stat_d, pv_d = chi_test(cnt_d, exact, N)
print(f"control (draft distribution) vs target:    chi2 = {stat_d:.1f}, p-value = {pv_d:.2e}")
assert pv > 0.001 and pv_d < 1e-6

speculative output vs target distribution: chi2 = 62.5, p-value = 0.867   (a large p-value = same distribution)
control (draft distribution) vs target:    chi2 = 3343149.9, p-value = 0.00e+00


**test 2, toy models: acceptance rate equals 1 - TV(p, q)**

In [32]:
st = dict(drafted=0, accepted=0)
for _ in range(N):
    _, s = speculative_decode(tp, tq, prefix, 1, 1, rng)
    st["drafted"] += s["drafted"]; st["accepted"] += s["accepted"]
tv = 0.5 * np.abs(Tp[0] - Tq[0]).sum()
print(f"measured acceptance rate {st['accepted'] / st['drafted']:.4f}   vs   1 - TV(p, q) = {1 - tv:.4f}")

measured acceptance rate 0.7004   vs   1 - TV(p, q) = 0.7034


**test 3, real models (Qwen2.5-0.5B draft, Qwen2.5-1.5B target, same tokenizer): one step on a real vocabulary**

In [33]:
from transformers import AutoModelForCausalLM, AutoTokenizer
t_name = "Qwen/Qwen2.5-1.5B"
target = AutoModelForCausalLM.from_pretrained(t_name, dtype=torch.float16).to(dev).eval()
draft = model                                                   # Qwen2.5-0.5B loaded at the top
assert AutoTokenizer.from_pretrained(t_name).get_vocab() == tok.get_vocab(), "tokenizers differ"
assert target.config.vocab_size == draft.config.vocab_size
print("shared tokenizer, vocabulary size", target.config.vocab_size)

config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

shared tokenizer, vocabulary size 151936


In [34]:
prompt = "Câu hỏi: Thủ đô của Việt Nam là thành phố nào?\nĐáp án:"
ids = tok(prompt, return_tensors="pt").input_ids.to(dev)
with torch.no_grad():
    p = torch.softmax(target(ids).logits[0, -1].double(), -1)
    q = torch.softmax(draft(ids).logits[0, -1].double(), -1)

def one_step(p, q, N, g):
    x = torch.multinomial(q, N, replacement=True, generator=g)                       # draft proposals
    u = torch.rand(N, device=p.device, generator=g, dtype=torch.float64)
    acc = u < torch.clamp(p[x] / q[x], max=1.0)                                      # accept with min(1, p/q)
    r = torch.clamp(p - q, min=0); r = r / r.sum()                                   # residual distribution (p - q)+
    y = torch.where(acc, x, torch.multinomial(r, N, replacement=True, generator=g))
    return y, acc.double().mean().item()

def chi_vs(y, p, N, min_exp=20):
    counts = torch.bincount(y, minlength=p.numel()).double()
    exp = N * p
    keep = exp >= min_exp
    stat = ((counts[keep] - exp[keep]) ** 2 / exp[keep]).sum() + (counts[~keep].sum() - exp[~keep].sum()) ** 2 / exp[~keep].sum()
    dof = int(keep.sum().item())                                                      # kept bins + 1 pooled bin - 1
    return stat.item(), chi2.sf(stat.item(), dof), int(keep.sum().item())

N = 200_000
g = torch.Generator(device=dev).manual_seed(0)
y, acc = one_step(p, q, N, g)
stat, pv, nb = chi_vs(y, p, N)
print(f"speculative step vs target distribution over {nb} tokens: chi2 = {stat:.1f}, p-value = {pv:.3f}")
y_q = torch.multinomial(q, N, replacement=True, generator=g)
stat_q, pv_q, _ = chi_vs(y_q, p, N)
print(f"control (draft only) vs target:                         chi2 = {stat_q:.1f}, p-value = {pv_q:.2e}")
tv = 0.5 * (p - q).abs().sum().item()
print(f"acceptance rate {acc:.4f}   vs   1 - TV(p, q) = {1 - tv:.4f}")
top = p.topk(5)
print("top-5 target tokens:", [(tok.decode([i]), round(v, 4), round((y == i).double().mean().item(), 4)) for v, i in zip(top.values.tolist(), top.indices.tolist())], "(token, p, empirical)")

speculative step vs target distribution over 133 tokens: chi2 = 135.0, p-value = 0.436
control (draft only) vs target:                         chi2 = 145405.1, p-value = 0.00e+00
acceptance rate 0.8285   vs   1 - TV(p, q) = 0.8285
top-5 target tokens: [(' Thủ', 0.6101, 0.6096), (' Thành', 0.1309, 0.1315), (' Hà', 0.1012, 0.1016), (' thủ', 0.016, 0.0161), (' T', 0.0105, 0.0106)] (token, p, empirical)


**full loop on the real pair: generate text and report the acceptance statistics**

In [35]:
def hf_probs(m):
    def f(ids):
        with torch.no_grad():
            lg = m(torch.tensor([ids], device=dev)).logits[0].float()
        return torch.softmax(lg, -1).cpu().numpy()
    return f

prompt_ids = tok("Thủ đô của Việt Nam là", return_tensors="pt").input_ids[0].tolist()
rng = np.random.default_rng(0)
out_ids, s = speculative_decode(hf_probs(target), hf_probs(draft), prompt_ids, 30, 4, rng)
print(tok.decode(out_ids))
print(f"target calls {s['target_calls']} | drafted {s['drafted']} | accepted {s['accepted']} | "
      f"acceptance {s['accepted'] / s['drafted']:.2f} | new tokens per target call {30 / s['target_calls']:.2f}")

Thủ đô của Việt Nam là đâu? Là kinh tế cổ điển hay hiện đại?
Thủ đô Việt Nam là Ekdep, Long An, Thanh Hoá hay Hà
target calls 11 | drafted 44 | accepted 19 | acceptance 0.43 | new tokens per target call 2.73
